# Zero-Shot Inference — NCTB MCQ
Runs vanilla zero-shot MCQ evaluation on the test split.
Results saved per model to `data/results/`.

**Run Cell 0 only on Colab/Kaggle. Skip it if running locally.**

In [ ]:
# ── Cell 0: Colab / Kaggle setup (skip if running locally) ──────────────────
# Clones the repo and installs dependencies. Run once per session.
import subprocess, os

GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <-- replace
REPO_NAME   = "BengaliLLM"             # <-- replace if different

if not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"], check=True)

os.chdir(REPO_NAME)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Setup done. CWD:", os.getcwd())

In [ ]:
import sys, os, json
import pandas as pd
from tqdm import tqdm

# Works for both local (notebooks/) and Colab/Kaggle (repo root as CWD)
cwd = os.getcwd()
REPO_ROOT = cwd if os.path.exists(os.path.join(cwd, 'src')) else os.path.abspath(os.path.join(cwd, '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

TEST_PATH   = os.path.join(REPO_ROOT, 'data', 'processed', 'test.jsonl')
RESULTS_DIR = os.path.join(REPO_ROOT, 'data', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Repo root:', REPO_ROOT)
print('Test     :', TEST_PATH)

## Cell 1 — Load test split

In [ ]:
# Loads the held-out test.jsonl into df_test
# Each row: id, domain, question, choices (dict), answer, source
df_test = pd.read_json(TEST_PATH, lines=True)
print('Test size:', len(df_test))
print('Domains  :', df_test['domain'].value_counts().to_dict())
df_test.head(2)

## Cell 2 — Inspect a sample prompt

In [ ]:
from src.evaluation.prompt import build_prompt

# Verify what the model will actually receive for a single row
sample_msgs = build_prompt(df_test.iloc[0].to_dict(), use_system_prompt=True)
for msg in sample_msgs:
    print(f"[{msg['role'].upper()}]\n{msg['content']}\n")

## Cell 3 — Choose model to evaluate

In [ ]:
from src.evaluation.config import MODEL_CONFIGS

# Change MODEL_ID and re-run Cells 4-8 for each model
# Available keys:
#   "Qwen/Qwen3-8B"
#   "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
#   "md-nishat-008/TigerLLM-9B-it"
#   "md-nishat-008/TigerLLM-1B-it"
#   "hishab/titulm-llama-3.2-3b-v2.0"
MODEL_ID = "Qwen/Qwen3-8B"   # <-- change this per run

cfg = MODEL_CONFIGS[MODEL_ID]
IS_REASONING_MODEL = "R1" in MODEL_ID or "r1" in MODEL_ID
print('Model            :', MODEL_ID)
print('Temperature      :', cfg['temperature'])
print('Use system prompt:', cfg['use_system_prompt'])
print('Max new tokens   :', cfg['max_new_tokens'])
print('Reasoning model  :', IS_REASONING_MODEL)

## Cell 4 — Load model and tokenizer

In [ ]:
from src.evaluation.inference import load_model_and_tokenizer

# Downloads weights on first run (cached by HuggingFace after that)
# bfloat16 + device_map='auto' handles single and multi-GPU setups
model, tokenizer = load_model_and_tokenizer(MODEL_ID)
print('Device map:', model.hf_device_map if hasattr(model, 'hf_device_map') else 'single device')

## Cell 5 — Run zero-shot inference

In [ ]:
from src.evaluation.inference import generate_response
from src.evaluation.prompt    import build_prompt

# Iterates over all test rows; appends raw model output per row
# R1 outputs a long <think>...</think> chain before the final answer
raw_outputs = []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=MODEL_ID):
    messages = build_prompt(row.to_dict(), use_system_prompt=cfg['use_system_prompt'])
    output   = generate_response(
        messages, model, tokenizer,
        temperature=cfg['temperature'],
        max_new_tokens=cfg['max_new_tokens']
    )
    raw_outputs.append(output)

df_test = df_test.copy()   # avoid SettingWithCopyWarning
df_test['raw_output'] = raw_outputs
print('Sample output:', raw_outputs[0][:200])

## Cell 6 — Extract predictions from raw output

In [ ]:
from src.evaluation.extractor import extract_prediction

# Extracts A/B/C/D from raw output via regex
# prediction=None → model didn't produce a valid letter (counted as invalid)
df_test['prediction'] = df_test['raw_output'].apply(
    lambda x: extract_prediction(x, is_reasoning_model=IS_REASONING_MODEL)
)
print('Prediction distribution:')
print(df_test['prediction'].value_counts(dropna=False).to_dict())

## Cell 7 — Compute and display metrics

In [ ]:
from src.evaluation.metrics import compute_metrics

# overall_acc and domain_acc are computed only over valid predictions
# invalid_rate tells you how often the model didn't return A/B/C/D
metrics = compute_metrics(df_test)

print(f"Model           : {MODEL_ID}")
print(f"Total samples   : {metrics['total']}")
print(f"Valid responses : {metrics['valid']}")
print(f"Invalid rate    : {metrics['invalid_rate']:.2%}")
print(f"Overall Acc     : {metrics['overall_acc']:.2%}")
print("\nPer-domain accuracy:")
for domain, acc in metrics['domain_acc'].items():
    print(f"  {domain:20s}: {acc:.2%}")

## Cell 8 — Save results

In [ ]:
# Saves predictions CSV and metrics JSON, both keyed by model name
safe_name    = MODEL_ID.replace('/', '__')
csv_path     = os.path.join(RESULTS_DIR, f'{safe_name}.csv')
metrics_path = os.path.join(RESULTS_DIR, f'{safe_name}_metrics.json')

df_test.to_csv(csv_path, index=False)
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({'model': MODEL_ID, **metrics}, f, ensure_ascii=False, indent=2)

print('Saved:', csv_path)
print('Saved:', metrics_path)

## Cell 9 — Compare all models (run after all models complete)

In [ ]:
import glob

# Aggregates all *_metrics.json files into a single comparison DataFrame
metric_files = glob.glob(os.path.join(RESULTS_DIR, '*_metrics.json'))

rows = []
for f in metric_files:
    with open(f) as fp:
        m = json.load(fp)
    rows.append({
        'model':        m['model'],
        'overall_acc':  m['overall_acc'],
        'invalid_rate': m['invalid_rate'],
        **{f"acc_{k}": v for k, v in m.get('domain_acc', {}).items()}
    })

df_compare = pd.DataFrame(rows).sort_values('overall_acc', ascending=False)
df_compare